In [1]:
import requests
from bs4 import BeautifulSoup
import logging
from datetime import datetime
import csv


In [2]:

PORTAL = "ekantipur"
URL = "https://ekantipur.com/sitemap_index/newsdetail"

headers = {
    "User-Agent": "Mozilla/5.0"
}

def fetch_page(url):
    try:
        logging.info(f"Fetching: {url}")
        res = requests.get(url, headers=headers, timeout=10)
        res.raise_for_status()
        return res.text
    except Exception as e:
        logging.error(f"Request failed: {e}")
        return None


In [3]:

def recent_articles_index():
    res = fetch_page(URL)

    if not res:
        return []

    soup = BeautifulSoup(res, "xml")

    articles = []
    today = datetime.now().date()

    for url in soup.find_all("url"):
        loc = url.find("loc")
        news = url.find("news:news")

        if not loc or not news:
            continue

        # language filter
        language = news.find("news:language")
        if not language or language.text.strip() != "ne":
            continue

        pub_date_el = news.find("news:publication_date")
        if not pub_date_el:
            continue

        title = news.find("news:title")

        articles.append({
            "link": loc.text.strip(),
            "title": title.text.strip() if title else None,
            "posted_at": pub_date_el.text.strip(),
            "source": PORTAL
        })

    logging.info(f"Extracted {len(articles)} Nepali articles")
    return articles


def save_to_csv(articles, filename):
    if not articles:
        logging.info("No articles to save.")
        return

    fieldnames = ["title", "link", "posted_at", "source"]

    try:
        with open(filename, mode="w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(articles)

        logging.info(f"Saved {len(articles)} articles to {filename}")

    except Exception as e:
        logging.error(f"Failed to save CSV: {e}")


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)

    articles = recent_articles_index()

    filename = f"ekantipur_articles_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    save_to_csv(articles, filename)

    print(f"Saved to {filename}")

INFO:root:Fetching: https://ekantipur.com/sitemap_index/newsdetail
INFO:root:Extracted 500 Nepali articles
INFO:root:Saved 500 articles to ekantipur_articles_20260426_170237.csv


Saved to ekantipur_articles_20260426_170237.csv
